# Lesson 1: Introduction to Ollama

* Author: [WhiteWaxula](mailto:whitewaxula@proton.me)
* License: MIT

> **Side Note**
>
> If you found this piece of content useful, feel free to star the repository and/or give me credit. It is very much appreciated.

## 1. Motivation and context

Ollama is a free, open-source platform that helps download and run LLMs directly on your computer. While running local models is the typical use case for ollama, you can still use it to access remote ollama servers as long as the endpoint is known.

Because ollama helps you run AI models locally, it has a series of advantages:

* Privacy: Your prompts and conversations will stay in your machine. You can even make sure of this by turning off your Internet connection when using ollama, your model inferences will still work even in that scenario.
* "No Cost": You won't have to pay for tokens, as it happens with other options. However, "No Cost" is not fully accurate, as operational costs will exist (electricity for your PC for example).
* Offline access: You can access models fully offline 
* Ease of use: Ollama is a fairly simple framework to use, and provides you with the required tools to scale towards more complex solutions as well.

## 2. Installation, starting and stopping Ollama

You can install Ollama following the instructions available on the official webpage. As of writing this notebook, these instructions are hosted on the [Ollama download page](https://ollama.com/download/).

After following the official instructions, you will be able to use the Ollama CLI. You can check the correct installation of Ollama by running the following command:

```console
$ ollama --version
ollama version is 0.33.3
```

If any version number is displayed, you'll have confirmation that Ollama was correctly installed on your system.

By default, Ollama will create a server to serve LLMs (for further usage) on startup. If you want to disable that behaviour (i.e., if you want to decide when the server will be started), simply disable Ollama at startup:

```console
$ sudo systemctl disable ollama
Removed '/etc/systemd/system/default.target.wants/ollama.service'.
```

Some other solutions exist, as they have been discussed in the official repository over time.

If you want to start the Ollama server, you'll need to run the following command:

```console
$ ollama serve
```

Alternatively, you could start the Ollama service:

```console
$ sudo systemctl start ollama
```

Similarly, if you want to stop it:

1. If started as a service:

    ```console
    $ sudo systemctl stop ollama
    ```

2. If started via the command line (i.e., using ollama serve), simply kill the process by pressing Ctrl+C.

## 3. Listing and downloading models

In order to list your locally available models, you can run the following command:

```console
$ ollama list
NAME          ID              SIZE      MODIFIED
```

Available models are listed in the [Ollama Library webpage](https://ollama.com/library). If you click in any of them and copy the identifier (e.g, an example identifier for Qwen 2.5 7B `qwen2.5:7b`) you will be able to download them locally:

```console
$ ollama pull qwen2.5:7b
```

You can then re-execute the list command to see that the model is fully available:

```console
$ ollama list
NAME          ID              SIZE      MODIFIED     
qwen2.5:7b    845dbda0ea48    4.7 GB    15 hours ago  
```

In Linux operating systems, models will be downloaded in `/usr/share/ollama/.ollama/models/`.

If you want to remove a model, you can simply run the `ollama rm` command, just like showcased next:

```console
$ ollama rm qwen2.5:7b
deleted 'qwen2.5:7b'
```

## 4. Running models

Ollama allows the user to interact with models in different ways:

1. Using the Ollama CLI. This
2. Using the `ollama` python package available in [PyPi](https://pypi.org/project/ollama/).
3. Sending requests to the Ollama Server (e.g., using the `requests` module to send requests to the ollama server).

All three methods allow for sending requests locally and to a remote ollama server. This is mainly done via an environment variable called `OLLAMA_HOST`, although this will be discussed in later lessons.

### 4.1. Ollama CLI

To run a model with ollama cli, we can simply use the following command:

```console
$ ollama run qwen2.5:7b
>>> Send a message (/? for help)
```

The user will then be able to either run commands (use `/?` for help) or send prompts to the model:

```console
>>> Say "Hello world"
Hello world! How can I assist you today?
>>> Send a message (/? for help)
```

Of course the output you get will depend on the model you're running. For this example, we sticked to [Qwen2.5:7b](https://ollama.com/library/qwen2.5), which is quite a fast model compared to others.

You can check if a model is running by simply using the following command:

```console
$ ollama ps
NAME          ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen2.5:7b    845dbda0ea48    5.1 GB    100% CPU     4096       3 minutes from now
```

You can stop a specific model by running the following command:

```console
$ ollama stop qwen2.5:7b
```

> **NOTE**
> 
> Stopping a model is more important than it seems. If we exit the model CLI (typing `/exit`), the model will still be loaded. This can be confirmed by simply running the `$ ollama ps` command. While that can be useful in some scenarios, you need to be conscious of the resources consumed and available, as you may want to liberate them for other tasks.

### 4.2. Python package

In order to use the `ollama` python package, it is recommendable to first create a virtual environment. The virtual environment will help avoid polluting the global python environment with project-specific dependencies. This can be done in many ways, for this tutorial we will use [`uv`](https://docs.astral.sh/uv/):

```console
$ uv init
$ uv venv
$ source .venv/bin/activate
```

Next, we will add the ollama dependency (see [ollama in PyPi](https://pypi.org/project/ollama/) for more information):

```console
$ uv add ollama
```

Upon installation, the following code will allow us to prompt a specific model. Do not feel scared by the syntax, future lessons will help you learn more about it!

In [1]:
from ollama import generate

MODEL_ID = "qwen2.5:7b"
PROMPT = "Hello world!"

output = generate(MODEL_ID, PROMPT)
print(f"User: '{PROMPT}'")
print(f"{MODEL_ID}: {output["response"]}")

User: 'Hello world!'
qwen2.5:7b: Hello! How can I assist you today?


## 4.3. API Request

Similarly, we can send a prompt to the model by directly using the corresponding endpoint of the API (we'll get into that in later sessions). The API specification is shown in the [official webpage](https://docs.ollama.com/api/), but it simply provides endpoints for the main functionalities, including generation, chat (i.e., with the ability to provide relevant context and previous messages), embeddings, model download, and so on.

The following code snippet simply uses Python `requests` module to run a `POST` request on the locally hosted ollama server (keeping the same model and prompt as before):

In [2]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_ID, "prompt": PROMPT, "stream": False}
)
response.raise_for_status()

print(f"Response status: {response.status_code}")

Response status: 200


As can be seen, the response is completely successful, we obtain a 200 OK response from the ollama server. Let us process the response to obtain the output generated by the model:

In [3]:
output = response.json()
print(f"User: '{PROMPT}'")
print(f"{MODEL_ID}: {output["response"]}")

User: 'Hello world!'
qwen2.5:7b: Hello! Nice to meet you. Is there anything specific you'd like to talk about or any questions you have?
